In [5]:
# Cell 1 — imports
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

raw = Path("../data/raw")

train = pd.read_csv(raw / "crop_yield_train.csv")
test = pd.read_csv(raw / "crop_yield_leaderboard_test.csv")
template = pd.read_csv(raw / "submission_template.csv")
weather = pd.read_csv(raw / "regional_weather.csv")
prices = pd.read_csv(raw / "market_prices.csv")

print("train", train.shape)
print("test", test.shape)
print("template", template.shape)
print("weather", weather.shape)
print("prices", prices.shape)

print("\nTrain columns:", list(train.columns))
print("\nTest columns:", list(test.columns))
print("\nWeather columns:", list(weather.columns))
print("\nPrices columns:", list(prices.columns))

train (15090, 15)
test (3750, 14)
template (3750, 2)
weather (232, 6)
prices (100, 4)

Train columns: ['plot_id', 'region', 'crop_type', 'survey_year', 'planting_month', 'altitude_m', 'rainfall_mm_season', 'farm_size_ha', 'fertilizer_kg_per_ha', 'improved_seed_used', 'pest_disease_flag', 'soil_quality_index', 'labor_days_per_ha', 'distance_to_market_km', 'yield_tons_per_ha']

Test columns: ['plot_id', 'region', 'crop_type', 'survey_year', 'planting_month', 'altitude_m', 'rainfall_mm_season', 'farm_size_ha', 'fertilizer_kg_per_ha', 'improved_seed_used', 'pest_disease_flag', 'soil_quality_index', 'labor_days_per_ha', 'distance_to_market_km']

Weather columns: ['region', 'year', 'month', 'avg_temp_c', 'monthly_rainfall_mm', 'extreme_heat_days']

Prices columns: ['crop_type', 'region', 'year', 'price_birr_per_quintal']


In [7]:
mean_yield = train["yield_tons_per_ha"].mean()
print("Mean yield:", mean_yield)

sub = template.copy()
sub["predicted_yield_tons_per_ha"] = mean_yield

out = Path("../submission/team_dawit_submission.csv")
sub.to_csv(out, index=False)

assert len(sub) == 3750, "Wrong number of rows"
assert list(sub.columns) == ["plot_id", "predicted_yield_tons_per_ha"]
assert sub["plot_id"].is_unique
assert sub["predicted_yield_tons_per_ha"].notna().all()
print("Baseline submission saved and validated:", out)

Mean yield: 2.762496745973244
Baseline submission saved and validated: ..\submission\team_dawit_submission.csv


A1 Cleaning Log


In [8]:
def clean_region(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower()
    if "oromia" in x: return "Oromia"
    if "amhara" in x: return "Amhara"
    if "snnpr" in x: return "SNNPR"
    if "tigray" in x: return "Tigray"
    if "somali" in x: return "Somali"
    return x.title()

def clean_crop(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower()
    if x in ["teff", "tef"]: return "teff"
    if "wheat" in x: return "wheat"
    if "maize" in x: return "maize"
    if "sorghum" in x: return "sorghum"
    if "barley" in x: return "barley"
    return x

# Show before
print("Before region train:", sorted(train["region"].dropna().unique()))
print("Before crop train:", sorted(train["crop_type"].dropna().unique()))
print("Before region weather:", sorted(weather["region"].dropna().unique()))
print("Before region prices:", sorted(prices["region"].dropna().unique()))
print("Before crop prices:", sorted(prices["crop_type"].dropna().unique()))

for df in [train, test, weather, prices]:
    if "region" in df.columns:
        df["region"] = df["region"].apply(clean_region)
    if "crop_type" in df.columns:
        df["crop_type"] = df["crop_type"].apply(clean_crop)

# Show after
print("\nAfter region train:", sorted(train["region"].dropna().unique()))
print("After crop train:", sorted(train["crop_type"].dropna().unique()))
print("After region weather:", sorted(weather["region"].dropna().unique()))
print("After region prices:", sorted(prices["region"].dropna().unique()))
print("After crop prices:", sorted(prices["crop_type"].dropna().unique()))

Before region train: ['AMHARA', 'Amhara', 'Amhara ', 'OROMIA', 'Oromia', 'Oromia ', 'SNNPR', 'SNNPR ', 'SOMALI', 'Somali', 'Somali ', 'TIGRAY', 'Tigray', 'Tigray ', 'amhara', 'oromia', 'snnpr', 'somali', 'tigray']
Before crop train: [' barley ', ' maize ', ' sorghum ', ' teff ', ' wheat ', 'BARLEY', 'Barley', 'MAIZE', 'Maize', 'SORGHUM', 'Sorghum', 'TEFF', 'Tef', 'Teff', 'WHEAT', 'Wheat', 'barley', 'maize', 'sorghum', 'teff', 'wheat']
Before region weather: ['AMH', 'Amhara', 'Amhara ', 'ORO', 'Oromia', 'Oromia ', 'SNNP', 'SNNPR', 'SNNPR ', 'SOM', 'Somali', 'Somali ', 'TIG', 'Tigray', 'Tigray ', 'amhara', 'oromia', 'snnpr', 'somali', 'tigray']
Before region prices: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
Before crop prices: [' barley ', ' maize ', ' sorghum ', ' teff ', ' wheat ', 'Barley', 'Maize', 'Sorghum', 'Wheat', 'barley', 'maize', 'sorghum', 'teff', 'wheat']

After region train: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
After crop train: ['barley', 'maize', 'sor

In [5]:
for df in [train, test]:
    df.replace([-999, -999.0, -999.00], np.nan, inplace=True)
    

In [6]:
print(train.isna().sum())
print(train[["farm_size_ha","fertilizer_kg_per_ha","distance_to_market_km","rainfall_mm_season","altitude_m"]].describe())


plot_id                    0
region                     0
crop_type                  0
survey_year                0
planting_month             0
altitude_m                 0
rainfall_mm_season       449
farm_size_ha               0
fertilizer_kg_per_ha     748
improved_seed_used         0
pest_disease_flag        300
soil_quality_index       608
labor_days_per_ha        488
distance_to_market_km      0
yield_tons_per_ha          0
dtype: int64
       farm_size_ha  fertilizer_kg_per_ha  distance_to_market_km  \
count  15090.000000          14342.000000           15090.000000   
mean       1.399811             41.661507              11.902898   
std        2.459939             47.201509              11.847415   
min        0.129617              0.498323               0.500000   
25%        0.756104             19.706949               3.434791   
50%        1.095662             33.827426               8.231581   
75%        1.597363             53.225263              16.429947   
max     

In [7]:
caps = {
    "farm_size_ha": train["farm_size_ha"].quantile(0.99),
    "fertilizer_kg_per_ha": train["fertilizer_kg_per_ha"].quantile(0.99),
    "distance_to_market_km": train["distance_to_market_km"].quantile(0.99),
    "rainfall_mm_season": train["rainfall_mm_season"].quantile(0.99),
    "altitude_m": train["altitude_m"].quantile(0.99),
}
print(caps)

for col, cap in caps.items():
    train[col] = train[col].clip(upper=cap)
    test[col] = test[col].clip(upper=cap)

{'farm_size_ha': np.float64(4.262495503276447), 'fertilizer_kg_per_ha': np.float64(136.04985626055972), 'distance_to_market_km': np.float64(54.93301599047047), 'rainfall_mm_season': np.float64(1406.0986457926153), 'altitude_m': np.float64(3100.0)}


A3 -join map


In [8]:
month_map = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
             "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}

for df in [train, test]:
    df["planting_month_num"] = (
        df["planting_month"].astype(str).str.strip().str[:3].map(month_map)
    )

weather["month_num"] = (
    weather["month"].astype(str).str.strip().str[:3].map(month_map)
)

# Numeric coercion
for c in ["avg_temp_c","monthly_rainfall_mm","extreme_heat_days"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

# Drop duplicate weather readings (region, year, month) keeping first
before = len(weather)
weather = weather.drop_duplicates(subset=["region","year","month_num"], keep="first")
print("Weather rows before/after dedup:", before, len(weather))

Weather rows before/after dedup: 232 226


In [9]:
def make_season_rows(df):
    rows = []
    for _, r in df.iterrows():
        for off in range(4):
            m = r["planting_month_num"] + off
            y = r["survey_year"] + (m - 1) // 12
            mm = (m - 1) % 12 + 1
            rows.append({"plot_id": r["plot_id"], "region": r["region"],
                         "year": y, "month_num": mm})
    return pd.DataFrame(rows)

season_train = make_season_rows(train)
season_test  = make_season_rows(test)

weather_small = weather[["region","year","month_num",
                         "avg_temp_c","monthly_rainfall_mm","extreme_heat_days"]]

season_train = season_train.merge(weather_small, on=["region","year","month_num"], how="left")
season_test  = season_test.merge(weather_small, on=["region","year","month_num"], how="left")

def agg_weather(season):
    return season.groupby("plot_id").agg(
        season_mean_temp=("avg_temp_c","mean"),
        season_total_rain=("monthly_rainfall_mm","sum"),
        season_extreme_heat=("extreme_heat_days","sum"),
        season_months_count=("avg_temp_c","count"),
    ).reset_index()

agg_train = agg_weather(season_train)
agg_test  = agg_weather(season_test)

train = train.merge(agg_train, on="plot_id", how="left")
test  = test.merge(agg_test, on="plot_id", how="left")

print("train shape after weather join:", train.shape)
print("test shape after weather join:", test.shape)

train shape after weather join: (15090, 20)
test shape after weather join: (3750, 19)


A4- audit

A6-Feature engineering table


In [10]:
reg_year = weather.groupby(["region","year"]).agg(
    region_year_temp=("avg_temp_c","mean"),
    region_year_rain=("monthly_rainfall_mm","sum"),
).reset_index()

train = train.merge(reg_year, left_on=["region","survey_year"],
                    right_on=["region","year"], how="left")
test  = test.merge(reg_year, left_on=["region","survey_year"],
                   right_on=["region","year"], how="left")

for df in [train, test]:
    df["temp_dev"] = df["season_mean_temp"] - df["region_year_temp"]
    df["rain_dev"] = df["season_total_rain"] - df["region_year_rain"]
    df["fert_x_seed"] = df["fertilizer_kg_per_ha"] * df["improved_seed_used"]

print(train[["season_mean_temp","season_total_rain","season_extreme_heat",
             "temp_dev","rain_dev","fert_x_seed"]].head())

   season_mean_temp  season_total_rain  season_extreme_heat  temp_dev  \
0         17.866667              205.9                  0.0 -0.406061   
1         15.466667              555.9                  0.0 -0.473333   
2         19.200000              294.9                  0.0  0.927273   
3         19.425000              398.3                  1.0 -0.385000   
4         16.433333              583.6                  1.0 -0.603030   

   rain_dev  fert_x_seed  
0    -701.5     0.000000  
1    -284.9    16.843992  
2    -612.5     0.000000  
3    -603.5     0.000000  
4    -630.0    33.736138  


In [14]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

submission_dir = PROJECT_ROOT / "submission"
submission_dir.mkdir(parents=True, exist_ok=True)
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Submission dir:", submission_dir)


Project root: c:\Users\sclab\Desktop\A hackaton
Submission dir: c:\Users\sclab\Desktop\A hackaton\submission


In [10]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

submission_dir = PROJECT_ROOT / "submission"
print("submission_dir:", submission_dir)

submission_dir: c:\Users\sclab\Desktop\A hackaton\submission


In [11]:
print("train regions:", sorted(train["region"].dropna().unique()))
print("Weather regions:", sorted(weather["region"].dropna().unique()))
print("Prices regions:", sorted(prices["region"].dropna().unique()))
print("Train crops:", sorted(train["crop_type"].dropna().unique()))
print("Prices crops:", sorted(prices["crop_type"].dropna().unique()))
print("Submission shape:", pd.read_csv(submission_dir / f"team_dawit_submission.csv").shape)

train regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
Weather regions: ['Amh', 'Amhara', 'Oro', 'Oromia', 'SNNPR', 'Snnp', 'Som', 'Somali', 'Tig', 'Tigray']
Prices regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
Train crops: ['barley', 'maize', 'sorghum', 'teff', 'wheat']
Prices crops: ['barley', 'maize', 'sorghum', 'teff', 'wheat']
Submission shape: (3750, 2)
